In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

#Now we will generate step by step all of the properties that
#effect Roman scale degrees in our dataset in order to create
#a generative model with stacking information that progresses
#with the full information (the term stacking was used, because)
#the information will stack in order to predict the next offset).

#First we need to compute the chord duration, in order to take advantage the
#fact that in each beat type based on the time different chords are being used
#in baroque harmony.
#the duration will be the time that one chord needs until the next one.
df["duration"] = df.groupby("chorale_name")["offset"].diff().shift(-1)

#the last chord of each chorale has no successor, so its duration is what
#remains until the end of its bar
last = df["duration"].isna()
df.loc[last, "duration"] = (df.loc[last, "bar_start_offset"]
                            + df.loc[last, "len"]
                            - df.loc[last, "offset"])

dur_values = sorted(df.duration.unique())
#mapping duration
dur_map = {dur_values[i]: i for i in range(len(dur_values))}
n_dur = len(dur_values)

cgroup = df.groupby("chorale_name").agg(list)

#spliting
rng = np.random.default_rng(25267220)
location = rng.permutation(len(cgroup))

#make a list for each predictor/target variable
roman     = cgroup["roman_scale_degree"].tolist()
qual      = cgroup["chord_quality"].tolist()
inversion = cgroup["inversion"].tolist()
beat      = cgroup["beat_strength"].tolist()
has7      = cgroup["has7"].tolist()
steps     = cgroup["steps_to_end"].tolist()
duration  = cgroup["duration"].tolist()

#we split the data into test and train and train
#with consistency about 14%, 14%, and, 72% accordingly
test_slice  = slice(0, 50)
val_slice   = slice(50, 100)
train_slice = slice(100, None)

#spliting using location
roman = [roman[i] for i in location]
qual = [qual[i] for i in location]
inversion = [inversion[i] for i in location]
beat = [beat[i] for i in location]
has7 = [has7[i] for i in location]
steps = [steps[i] for i in location]
duration = [duration[i] for i in location]

#Transforming the data (predictors and target variables) into a suited form
#for the model to process.
def make_windows(dslice, wsize):
  #For each predictor/target variable (they are the same)
  #we will turn the variable into a list (grouped by chorale name)
  #that each row contains
  #the list of the values of that variable for each chorale,
  #since we predict each time, the continuation of a chorale.
  Xr, Xq, Xi, Xh, Xb, Xs, Xd, Xbnow = [], [], [], [], [], [], [], []
  y_roman, y_qual, y_inv, y_h7, y_dur = [], [], [], [], []
  for r, q, i, b, h7, s, d in zip(roman[dslice], qual[dslice],
                              inversion[dslice], beat[dslice], has7[dslice],
                              steps[dslice], duration[dslice]):
    #We turn roman degrees and quality, has 7, duration into integers
    roman_map = [chord_map[j] for j in r]
    qual_map = [quality_map[j] for j in q]
    d_map = [dur_map[j] for j in d]
    h7_map = [int(j) for j in h7]
    for j in range(len(roman_map) - wsize):
      Xr.append(roman_map[j:j+wsize])
      Xq.append(qual_map[j:j+wsize])
      Xi.append(i[j:j+wsize])
      Xb.append(b[j:j+wsize])
      Xh.append(h7_map[j:j+wsize])
      Xs.append(s[j:j+wsize])
      Xd.append(d_map[j:j+wsize])
      #it will come from knowledge not as data leak, from the duration of the
      #previous chord
      Xbnow.append(b[j+wsize])

      #changing y based on the target variable - in this model there will be
      #more than one
      y_roman.append(roman_map[j+wsize])
      y_qual.append(qual_map[j+wsize])
      y_inv.append(i[j+wsize])
      y_h7.append(h7_map[j+wsize])
      y_dur.append(d_map[j+wsize])

  X = [np.array(Xr), np.array(Xq), np.array(Xi), np.array(Xh), np.array(Xd),
       np.array(Xb).reshape(-1, wsize, 1), np.array(Xs).reshape(-1, wsize, 1),
       np.array(Xbnow).reshape(-1, 1)]

  Y = [np.array(y_roman), np.array(y_qual), np.array(y_inv),
  np.array(y_h7), np.array(y_dur)]

  return X,Y

wsize,units = 8,128

X_train, Y_train = make_windows(train_slice, wsize)
X_val,   Y_val   = make_windows(val_slice,   wsize)
X_test,  Y_test  = make_windows(test_slice,  wsize)
print("X:", [x.shape for x in X_train])
print("Y:", [y.shape for y in Y_train])

#This time we will create a chained model to see if by using chained the
#predicted values as previous information for all target variables we will
#be able to get better results
Xr = layers.Input(shape = (wsize,), name = "roman.")
Xq = layers.Input(shape = (wsize,), name = "qual.")
Xi = layers.Input(shape = (wsize,), name = "inv.")
Xh = layers.Input(shape = (wsize,), name = "has7.")
Xd = layers.Input(shape = (wsize,), name = "dur.")
Xb = layers.Input(shape = (wsize,1), name = "beat.")
Xs = layers.Input(shape = (wsize,1), name = "steps.")
Xbnow = layers.Input(shape=(1,), name="beat_now.")

embroman = layers.Embedding(14, 16)(Xr)
embqual = layers.Embedding(5, 6)(Xq)
embinv = layers.Embedding(6, 8)(Xi)
embh7 = layers.Embedding(2, 4)(Xh)
embdur = layers.Embedding(n_dur, 8)(Xd)

x = layers.concatenate([embroman, embqual, embinv, embh7, embdur, Xb, Xs])
m_type = layers.LSTM(units)(x)

out_dur = layers.Dense(n_dur, activation="softmax", name="dur")(m_type)

m_type1 = layers.Concatenate()([m_type, out_dur, Xbnow])
out_roman = layers.Dense(len(chord_map), activation="softmax",
                         name="roman")(m_type1)

m_type2 = layers.Concatenate()([m_type1, out_roman])
out_qual = layers.Dense(len(quality_map), activation="softmax",
                         name="qual")(m_type2)

m_type3 = layers.Concatenate()([m_type2, out_qual])
out_inv = layers.Dense(6, activation="softmax", name="inv")(m_type3)

m_type4 = layers.Concatenate()([m_type3, out_inv])
out_h7    = layers.Dense(2, activation="softmax", name="has7")(m_type4)



model = keras.Model([Xr, Xq, Xi, Xh, Xd, Xb, Xs, Xbnow],
                    [out_roman, out_qual, out_inv, out_h7,
                     out_dur])

model.compile(optimizer = keras.optimizers.Adam(1e-3),
              loss=["sparse_categorical_crossentropy"] * 5,
              loss_weights=[1.0, 0.5, 0.5, 0.3, 0.5],
              metrics=[["accuracy"]] * 5)

early_stopping = keras.callbacks.EarlyStopping(patience=10,
                                               restore_best_weights=True)

history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                    epochs=50, batch_size=128, callbacks=[early_stopping],
                    verbose=1)

ev = model.evaluate(X_test, Y_test, verbose=0)
for name, value in zip(model.metrics_names, ev):
    print(f"{name:25} {value:.4f}")

In [ ]:
for name, x in zip(["roman","qual","inv","has7","dur","beat","steps","beat_now"], X_test):
    print(f"{name:10} shape {str(x.shape):18} first row: {x[0].ravel()[:9]}")

print("\ntargets, first 5 samples:")
for name, y in zip(["roman","qual","inv","has7","dur"], Y_test):
    print(f"{name:8} {y[:5]}")

# 2.Majority baseline is the score real or class imbalance affected results
print("\nmajority class share:")
for name, y in zip(["roman","qual","inv","has7","dur"], Y_test):
    print(f"{name:8} {pd.Series(y).value_counts(normalize=True).max():.3f}")

# Model Structure
model.summary()

#top 3 accuracy
#if the right chord number was part of the top 3 resuls = correct
probs = model.predict(X_test, verbose=0)[0]
top3 = np.argsort(-probs, axis=1)[:, :3]
print("top-3:", np.mean([y in t for y, t in zip(Y_test[0], top3)]))

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             balanced_accuracy_score, f1_score)

#getting the number of the different chords
n_roman = len(chord_map)
labels  = [k for k, v in sorted(chord_map.items(), key=lambda kv: kv[1])]

#The prior that the model has learned by training
prior = np.bincount(Y_train[0], minlength=n_roman).astype(float)
prior /= prior.sum()

#optimising tau
p_val = model.predict(X_val, verbose=0)[0]
best_tau, best_f1 = 0.0, -1.0
for tau in np.arange(0.0, 1.01, 0.05):
    pred = (p_val / (prior ** tau + 1e-12)).argmax(axis=1)
    f1 = f1_score(Y_val[0], pred, labels=range(n_roman),
                  average="macro", zero_division=0)
    if f1 > best_f1:
        best_tau, best_f1 = float(tau), f1
print(f"best tau = {best_tau:.2f}  (val macro-F1 = {best_f1:.4f})")

#adding the new results with tau at the test data
p_test = model.predict(X_test, verbose=0)[0]
y_true = Y_test[0]
y_raw  = p_test.argmax(axis=1)
y_pred = (p_test / (prior ** best_tau + 1e-12)).argmax(axis=1)

#adding accuracy, balanced accuracy and F1 macro score.
for tag, yp in (("raw", y_raw), (f"tau={best_tau:.2f}", y_pred)):
    print(f"\n--- {tag} ---")
    print(f"accuracy          {(yp == y_true).mean():.4f}")
    print(f"balanced accuracy {balanced_accuracy_score(y_true, yp):.4f}")
    print(f"macro F1          {f1_score(y_true, yp, labels=range(n_roman), average='macro', zero_division=0):.4f}")

#printing precision, recall, f1-score and number of instances for
#each unique chord.
print("\n" + classification_report(y_true, y_pred, labels=range(n_roman),
                                   target_names=labels, zero_division=0))

#confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=range(n_roman), normalize="true")
plt.figure(figsize=(13, 11))
ax = sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                 annot_kws={"size": 11},
                 xticklabels=labels, yticklabels=labels)
ax.set_xlabel("Predicted degree", fontsize=14)
ax.set_ylabel("True degree", fontsize=14)
ax.set_title(f"Confusion matrix — harmonic degree (tau = {best_tau:.2f})",
             fontsize=16, pad=14)
ax.tick_params(labelsize=12)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion.png", dpi=150, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
#The same analysis as the chunk before without tau changing priors.
print("\n" + "="*60)
print("RAW (no logit adjustment)")
print("="*60)
print(classification_report(y_true, y_raw, labels=range(n_roman),
                            target_names=labels, zero_division=0))

cm_raw = confusion_matrix(y_true, y_raw, labels=range(n_roman), normalize="true")
plt.figure(figsize=(13, 11))
ax = sns.heatmap(cm_raw, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                 annot_kws={"size": 11},
                 xticklabels=labels, yticklabels=labels)
ax.set_xlabel("Predicted degree", fontsize=14)
ax.set_ylabel("True degree", fontsize=14)
ax.set_title("Confusion matrix — harmonic degree (no prior correction)",
             fontsize=16, pad=14)
ax.tick_params(labelsize=12)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion_raw.png", dpi=150, bbox_inches="tight", transparent=True)
plt.show()

ev = model.evaluate(X_test, Y_test, verbose=0)
for name, value in zip(model.metrics_names, ev):
    print(f"{name:25} {value:.4f}")

In [ ]:
#computing and printing accuracy, top-3 accuracy, macroF1. balanced accuracy,
#CE, and PP for all target variables
preds = model.predict(X_test, verbose=0)
names = ["roman", "qual", "inv", "has7", "dur"]

print(f"{'target':8} {'top-1':>8} {'top-3':>8} {'macroF1':>9} {'balAcc':>8} {'CE':>7} {'PP':>7}")
for name, p, y in zip(names, preds, Y_test):
    yp   = p.argmax(axis=1)
    acc  = (yp == y).mean()
    top3 = np.mean([y[i] in np.argsort(p[i])[-3:] for i in range(len(y))])
    f1   = f1_score(y, yp, average="macro", zero_division=0)
    bal  = balanced_accuracy_score(y, yp)
    ce   = -np.mean(np.log(p[np.arange(len(y)), y] + 1e-12))
    print(f"{name:8} {acc:8.3f} {top3:8.3f} {f1:9.3f} {bal:8.3f} {ce:7.3f} {np.exp(ce):7.2f}")